# SFR Convex Screener Backtest

Event-driven backtest of `RVUtils/SFRConvexScreener.build_snapshot` over a `pd.bdate_range` of as_of dates.

Pipeline (mirror of `notebooks/backtests/sfr_fly_rv_backtest.ipynb`):
1. Build / load cached snapshots per as_of date.
2. Flatten snapshots into a `BacktestSignal` table.
3. Wire entry / exit triggers via `FlowSignalTriggerRequirements`.
4. Run `QueryDrivenBacktest` over the TimeGrid.
5. Tearsheet (MTM curve, drawdown, Sharpe, hit-rate, trade & exit log).

**Caveats** — see the bottom of the notebook.

In [ ]:
import sys
sys.path.append('../../')

import datetime
import logging
from pathlib import Path

import nest_asyncio
import numpy as np
import pandas as pd
import pytz
import matplotlib.pyplot as plt

nest_asyncio.apply()
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(name)s: %(message)s')

NYC = pytz.timezone('America/New_York')

## Config

In [ ]:
from RVUtils.SFRConvexScreener import (
    JointMethod,
    SFRConvexScreenerConfig,
    SFRScreenerBacktestConfig,
)
from RVUtils.SFRConvexScreener.backtest import run_backtest

screener_cfg = SFRConvexScreenerConfig(
    universe_size=12,
    include_outrights=True,
    jpm_method=True,
    primary_joint_method=JointMethod.HISTORICAL_GAUSSIAN_COPULA,
    correlation_window=60,
    n_simulations=50_000,
)

bt_cfg = SFRScreenerBacktestConfig(
    bpv_per_trade=100_000,
    entry_min_asymmetry=1.5,
    entry_min_composite_score=0.0,
    skip_stale=True,
    structure_types=('outright', 'calendar', 'butterfly'),
    max_concurrent=5,
    rebalance_dow=4,                 # Friday rebalance
    exit_asymmetry_threshold=1.10,
    exit_take_profit_bp=10.0,
    exit_stop_loss_bp=-15.0,
    exit_max_holding_days=22,
    round_trip_cost_bp=0.5,
)
screener_cfg, bt_cfg

## TimeGrid (6-month default)

In [ ]:
end = datetime.datetime(2026, 4, 28, 17, 0)
start = end - datetime.timedelta(days=180)
bdates = pd.bdate_range(NYC.localize(start), NYC.localize(end), tz=NYC)
bt_datetimes = [d.to_pydatetime() for d in bdates]
print(f'{len(bt_datetimes)} business datetimes from {bt_datetimes[0]} to {bt_datetimes[-1]}')

## Run

First run primes the on-disk cache; subsequent runs reuse it. Expect ~5-20 minutes per uncached as_of date.

In [ ]:
bt = run_backtest(
    bt_datetimes=bt_datetimes,
    screener_config=screener_cfg,
    backtest_config=bt_cfg,
    show_progress=True,
)
print(f'closed positions: {len(bt.portfolio.closed_positions_log)}')
print(f'mtm history: {len(bt.mtm_history)} datapoints')

## Tearsheet

In [ ]:
mtm_series = pd.Series(
    {pd.Timestamp(k): float(v) for k, v in bt.mtm_history.items()},
).sort_index()
if not mtm_series.empty:
    mtm_series.index = pd.DatetimeIndex(mtm_series.index).tz_localize(None)
mtm_series.head(), mtm_series.tail()

In [ ]:
if not mtm_series.empty:
    daily = mtm_series.diff().dropna()
    sharpe = float(np.sqrt(252) * daily.mean() / daily.std()) if daily.std() > 0 else float('nan')
    running_max = mtm_series.cummax()
    dd = (mtm_series - running_max)
    max_dd = float(dd.min())

    fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
    mtm_series.plot(ax=axes[0], title=f'Cumulative MTM  |  Sharpe={sharpe:.2f}')
    axes[0].set_ylabel('USD')
    dd.plot(ax=axes[1], title=f'Drawdown  |  max={max_dd:,.0f}', color='tab:red')
    axes[1].set_ylabel('USD')
    plt.tight_layout()
    plt.show()
    print(f'Sharpe={sharpe:.2f}  MaxDD={max_dd:,.0f}  Final={float(mtm_series.iloc[-1]):,.0f}')

## Trade log

In [ ]:
rows = []
for entry in bt.portfolio.closed_positions_log:
    pmeta = dict(entry.get('position_meta', {}) or {})
    xmeta = dict(entry.get('exit_meta', {}) or {})
    rows.append({
        'opened': entry['opened_at'],
        'closed': entry['closed_at'],
        'days': entry['holding_period_days'],
        'realized': entry['realized_pnl'],
        'gross': entry['gross_realized_pnl'],
        'fee': entry['fee_allocated'],
        'structure_id': pmeta.get('structure_id'),
        'structure_type': pmeta.get('structure_type'),
        'direction': pmeta.get('direction'),
        'entry_asymmetry': pmeta.get('entry_asymmetry'),
        'entry_composite': pmeta.get('entry_composite'),
        'exit_reason': xmeta.get('reason'),
    })
trades = pd.DataFrame(rows)
trades.head(20)

In [ ]:
if not trades.empty:
    print('--- exit reasons ---')
    print(trades['exit_reason'].value_counts())
    print('\n--- realized pnl by structure type ---')
    print(trades.groupby('structure_type')['realized'].agg(['count', 'mean', 'sum']))
    print(f'\nhit rate: {(trades["realized"] > 0).mean():.2%}')

In [ ]:
if not trades.empty:
    fig, ax = plt.subplots(figsize=(10, 4))
    for stype, sub in trades.groupby('structure_type'):
        ax.hist(sub['realized'].dropna().values, bins=20, alpha=0.5, label=str(stype))
    ax.legend()
    ax.set_title('Realized P&L by structure type')
    ax.set_xlabel('USD')
    plt.tight_layout()
    plt.show()

## Sensitivity (entry asymmetry threshold)

Cheap because the cache is shared — only the entry trigger filter changes.

In [ ]:
from dataclasses import replace

rows = []
for thresh in (1.5, 2.0, 3.0):
    cfg = replace(bt_cfg, entry_min_asymmetry=thresh)
    bt_i = run_backtest(
        bt_datetimes=bt_datetimes,
        screener_config=screener_cfg,
        backtest_config=cfg,
        show_progress=False,
    )
    mtm_i = pd.Series({pd.Timestamp(k).tz_localize(None) if pd.Timestamp(k).tzinfo else pd.Timestamp(k): float(v) for k, v in bt_i.mtm_history.items()}).sort_index()
    n = len(bt_i.portfolio.closed_positions_log)
    daily = mtm_i.diff().dropna()
    sharpe = float(np.sqrt(252) * daily.mean() / daily.std()) if daily.std() > 0 else float('nan')
    rows.append({
        'entry_min_asymmetry': thresh,
        'trades': n,
        'sharpe': sharpe,
        'final_mtm': float(mtm_i.iloc[-1]) if not mtm_i.empty else float('nan'),
    })
pd.DataFrame(rows)

## Caveats

- Snapshots are pickled into `data/screener_results/sfr_convex_screener_backtest_cache/`. The cache key hashes `_config_summary_for_cache` — bump that summary (or wipe the cache) if methodology changes (e.g. JPM method flag, gaps, joint method).
- `HISTORICAL_GAUSSIAN_COPULA` uses 60d daily-change correlation; in sparse-history regimes (post-launch contracts, tape gaps) the copula can collapse toward identity.
- Round-trip cost is a flat 0.5 bp per trade; tighten with venue-specific bid-ask before sizing live.
- The exit trigger's TP/SL path uses portfolio-level MTM (only meaningful with one open position). For multi-position portfolios, asymmetry-decay and max-hold dominate. Per-position MTM hooks would let TP/SL fire mid-portfolio.
- Default rolldown horizon is 1m — the SR3 IMM-IMM 3M schedule collapses on 3m so that horizon would be unreliable for outrights/calendars/flies.